# Issue: Broken Tool Call History in MemorySaver

复现生产环境中因 MemorySaver 累积了残缺 tool_call 历史而导致的两类 DeepSeek 400 错误：

| 类型 | 错误信息 | 原因 |
|------|----------|------|
| **A** | `insufficient tool messages following tool_calls` | AIMessage(tool_calls) 后面没有 ToolMessage |
| **B** | `tool must be response to a preceding message with tool_calls` | ToolMessage 前面没有 AIMessage(tool_calls) |

**为什么生产出现、notebook 不出现：**  
生产端同一个 `threadId` 被多次请求复用，第一次失败后 MemorySaver 存了坏状态，第二次请求就爆。  
notebook 每次 `new_thread()` 生成新 UUID，永远干净。

---

In [ ]:
import sys, os

AGENT_ROOT = '/data/home/lukemxjia/modelcraft/modelcraft-agent'
if AGENT_ROOT not in sys.path:
    sys.path.insert(0, AGENT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(AGENT_ROOT, '.env'))

from logging_setup import setup_logging
setup_logging()

import uuid
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from agents.admin_agent import admin_graph
from agents.shared import sanitize_messages

print('imports OK')

## 工具函数

In [ ]:
def make_state(layer='org', project_slug='', authorization='Bearer fake-token-for-test'):
    return {
        'messages'     : [],
        'authorization': authorization,
        'org_name'     : 'lukeco',
        'layer'        : layer,
        'project_slug' : project_slug,
        'current_model': '',
        'current_db'   : '',
    }

async def inject_bad_state(cfg: dict, bad_messages: list):
    """把坏消息序列直接写入 MemorySaver，模拟生产环境第一次失败后的残留状态。"""
    state = make_state()
    state['messages'] = bad_messages
    await admin_graph.aupdate_state(cfg, state)
    print(f'MemorySaver 已写入 {len(bad_messages)} 条坏消息')

async def run_agent(cfg: dict, user_msg: str = '我有哪些项目') -> str:
    """用同一个 thread_id 再次发请求，触发 MemorySaver 加载坏状态。"""
    state = make_state()
    state['messages'] = [HumanMessage(content=user_msg)]
    try:
        result = await admin_graph.ainvoke(state, config=cfg)
        reply = result['messages'][-1].content
        print(f'✅ 成功  Agent: {reply[:100]}')
        return reply
    except Exception as e:
        print(f'❌ 失败  {type(e).__name__}: {str(e)[:200]}')
        raise

---
## Case A：AIMessage(tool_calls) 后面没有 ToolMessage

错误：`insufficient tool messages following tool_calls message`

**触发场景**：第一次请求中 LLM 发起了 tool_call，但工具执行中途异常（如网络断开），  
ToolMessage 还没写入 MemorySaver 就崩了。

In [ ]:
print('=== Case A：复现 ===')
cfg_a = {'configurable': {'thread_id': str(uuid.uuid4())}}

# 坏状态：AIMessage 有 tool_calls，但后面没有 ToolMessage
bad_messages_a = [
    HumanMessage(content='我有哪些项目', id='human-1'),
    AIMessage(
        content='让我查询一下。',
        tool_calls=[{'name': 'list_projects', 'args': {}, 'id': 'call-a1', 'type': 'tool_call'}],
        id='ai-1'
    ),
    # ← 故意缺少 ToolMessage(tool_call_id='call-a1')
]

await inject_bad_state(cfg_a, bad_messages_a)

# 用同一 thread_id 发第二次请求 → 应该爆 400
try:
    await run_agent(cfg_a)
except Exception:
    print('✓ Case A 已复现')

In [ ]:
print('=== Case A：验证 sanitize_messages 修复 ===')

cleaned = sanitize_messages(bad_messages_a)
print(f'原始 {len(bad_messages_a)} 条 → 清理后 {len(cleaned)} 条')
for m in cleaned:
    tc = getattr(m, 'tool_calls', None)
    print(f'  {type(m).__name__}: {m.content[:50]!r}' + (f'  tool_calls={[t["name"] for t in tc]}' if tc else ''))

# AIMessage with tool_calls 应该被移除
has_orphan_ai = any(isinstance(m, AIMessage) and getattr(m, 'tool_calls', None) for m in cleaned)
assert not has_orphan_ai, 'FAIL: 仍有孤立 AIMessage(tool_calls)'
print('✅ Case A 修复验证通过')

---
## Case B：ToolMessage 前面没有 AIMessage(tool_calls)

错误：`Messages with role 'tool' must be a response to a preceding message with 'tool_calls'`

**触发场景**：第一版 `sanitize_messages` 把 AIMessage(tool_calls) 删掉了，  
但对应的 ToolMessage 没删干净，下一次 sanitize 时只剩孤立的 ToolMessage。

In [ ]:
print('=== Case B：复现 ===')
cfg_b = {'configurable': {'thread_id': str(uuid.uuid4())}}

# 坏状态：ToolMessage 没有对应的 AIMessage(tool_calls)
bad_messages_b = [
    HumanMessage(content='我有哪些项目', id='human-2'),
    # ← 故意缺少 AIMessage(tool_calls=[{id:'call-b1'}])
    ToolMessage(
        content="Error invoking tool 'list_projects' with kwargs {} with error:\n \n Please fix the error and try again.",
        tool_call_id='call-b1',
        name='list_projects',
        id='tool-1'
    ),
    AIMessage(content='抱歉，暂时无法获取项目列表。', id='ai-2'),
]

await inject_bad_state(cfg_b, bad_messages_b)

try:
    await run_agent(cfg_b)
except Exception:
    print('✓ Case B 已复现')

In [ ]:
print('=== Case B：验证 sanitize_messages 修复 ===')

cleaned = sanitize_messages(bad_messages_b)
print(f'原始 {len(bad_messages_b)} 条 → 清理后 {len(cleaned)} 条')
for m in cleaned:
    print(f'  {type(m).__name__}: {m.content[:60]!r}')

# 孤立 ToolMessage 应该被移除
has_orphan_tool = any(
    isinstance(m, ToolMessage) and getattr(m, 'tool_call_id', None) == 'call-b1'
    for m in cleaned
)
assert not has_orphan_tool, 'FAIL: 仍有孤立 ToolMessage'
print('✅ Case B 修复验证通过')

---
## Case C：完整的有效 pair，不应该被删除

In [ ]:
print('=== Case C：有效 pair 不应被删 ===')

valid_messages = [
    HumanMessage(content='我有哪些项目', id='human-3'),
    AIMessage(
        content='',
        tool_calls=[{'name': 'list_projects', 'args': {}, 'id': 'call-c1', 'type': 'tool_call'}],
        id='ai-3'
    ),
    ToolMessage(
        content='[{"id":"p1","slug":"abcde","title":"ABCDE"}]',
        tool_call_id='call-c1',
        name='list_projects',
        id='tool-3'
    ),
    AIMessage(content='你有以下项目：abcde', id='ai-4'),
    HumanMessage(content='帮我进入 abcde', id='human-4'),
]

cleaned = sanitize_messages(valid_messages)
assert len(cleaned) == len(valid_messages), f'FAIL: 有效消息被误删 {len(valid_messages)} → {len(cleaned)}'
print(f'✅ Case C 通过：{len(valid_messages)} 条均保留')

---
## 根本原因总结

```
同一 threadId 的生命周期（生产）：

  请求 1  → list_projects 失败（X-Action 错误 / 网络错误）
          → MemorySaver[threadId] = [..., AIMsg(tool_calls), ToolMsg(error)]  ← 坏状态 A
          或
          → MemorySaver[threadId] = [..., AIMsg(tool_calls)]                  ← 坏状态（更坏）

  请求 2（同 threadId）
          → ag_ui_langgraph.prepare_stream: state["messages"] = MemorySaver 历史
          → admin_agent.agent_node: messages = [system] + state["messages"] + [新 HumanMsg]
          → DeepSeek API 400

notebook 为什么不触发：
  new_thread() → 新 UUID → MemorySaver 为空 → 永远干净
```